# ETL Ques

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que1: SCD Type 2 — Slowly Changing Dimensions Query

**Difficulty:** Easy

### Problem

`customer_dim` is a Slowly Changing Dimension (SCD) Type 2 table: a customer can have several rows, each valid over the date range from `effective_date` to `end_date` (the current row uses `end_date` 9999-12-31). Return the version of each customer's record that was in effect on `2024-06-15` — the row whose validity range contains that date. For each, return `customer_id`, `name`, `city`, `effective_date`, and `end_date`.

**Schema columns:** `customer_dim.customer_id`, `customer_dim.name`, `customer_dim.city`, `customer_dim.effective_date`, `customer_dim.end_date`, `customer_dim.is_current`

**Output columns:** `customer_id`, `name`, `city`, `effective_date`, `end_date`

Sort the results by `customer_id` ascending.

### Examples

#### Example 1

**Input:**

**customer_dim:**

| customer_id | name | city | effective_date | end_date | is_current |
|------------:|------|------|----------------|----------|:----------:|
| 1 | John Smith | New York | 2024-01-01 | 2024-03-31 | 0 |
| 1 | John Smith | Los Angeles | 2024-04-01 | 9999-12-31 | 1 |
| 2 | Jane Doe | Chicago | 2024-01-15 | 2024-06-30 | 0 |
| 2 | Jane Doe | Boston | 2024-07-01 | 9999-12-31 | 1 |
| 3 | Bob Johnson | Seattle | 2024-02-01 | 2024-05-15 | 0 |
| 3 | Bob Johnson | Portland | 2024-05-16 | 9999-12-31 | 1 |
| 4 | Alice Williams | Denver | 2024-03-10 | 9999-12-31 | 1 |
| 5 | Charlie Brown | Austin | 2024-01-01 | 2024-04-30 | 0 |
| 5 | Charlie Brown | Houston | 2024-05-01 | 9999-12-31 | 1 |
| 6 | Diana Davis | Miami | 2024-06-01 | 9999-12-31 | 1 |
| 7 | Eve Martinez | Phoenix | 2024-02-15 | 2024-08-31 | 0 |
| 7 | Eve Martinez | Las Vegas | 2024-09-01 | 9999-12-31 | 1 |

**Output:**

| customer_id | name | city | effective_date | end_date |
|------------:|------|------|----------------|----------|
| 1 | John Smith | Los Angeles | 2024-04-01 | 9999-12-31 |
| 2 | Jane Doe | Chicago | 2024-01-15 | 2024-06-30 |
| 3 | Bob Johnson | Portland | 2024-05-16 | 9999-12-31 |
| 4 | Alice Williams | Denver | 2024-03-10 | 9999-12-31 |
| 5 | Charlie Brown | Houston | 2024-05-01 | 9999-12-31 |
| 6 | Diana Davis | Miami | 2024-06-01 | 9999-12-31 |
| 7 | Eve Martinez | Phoenix | 2024-02-15 | 2024-08-31 |

**Explanation:** Per customer, only the row whose date range covers 2024-06-15 is kept.

### Constraints

- The as-of date is `2024-06-15`.
- A row qualifies when `2024-06-15` falls on or between its `effective_date` and `end_date`, with both bounds inclusive.
- Do not include `is_current` in the output.
- Return results matching the expected output schema and order.

In [0]:
customer_dim_data = [(1,"John Smith","New York","2024-01-01","2024-03-31",0),(1,"John Smith","Los Angeles","2024-04-01","9999-12-31",1),(2,"Jane Doe","Chicago","2024-01-15","2024-06-30",0),(2,"Jane Doe","Boston","2024-07-01","9999-12-31",1),(3,"Bob Johnson","Seattle","2024-02-01","2024-05-15",0),(3,"Bob Johnson","Portland","2024-05-16","9999-12-31",1),(4,"Alice Williams","Denver","2024-03-10","9999-12-31",1),(5,"Charlie Brown","Austin","2024-01-01","2024-04-30",0),(5,"Charlie Brown","Houston","2024-05-01","9999-12-31",1),(6,"Diana Davis","Miami","2024-06-01","9999-12-31",1),(7,"Eve Martinez","Phoenix","2024-02-15","2024-08-31",0),(7,"Eve Martinez","Las Vegas","2024-09-01","9999-12-31",1)]
customer_dim_df = spark.createDataFrame(customer_dim_data, ["customer_id","name","city","effective_date","end_date","is_current"])

customer_dim_df = customer_dim_df.withColumn("effective_date", to_date(col("effective_date"), "yyyy-MM-dd")).withColumn("end_date", to_date(col("end_date"), "yyyy-MM-dd"))

display(customer_dim_df)

output_df = (
customer_dim_df
    .filter((col("effective_date") <= "2024-06-15" )  & (col("end_date") >= "2024-06-15"))
    .select("customer_id", "name",  "city", "effective_date", "end_date")
    .orderBy("customer_id")
)

display(output_df)



## Que2: ETL Data Cleaning Pipeline

**Difficulty:** Hard

### Problem

An e-commerce platform ingests raw order records that often arrive with quality problems: the same order can appear more than once, emails may be malformed, quantities can be negative, prices may be missing, and some dates fall in the future. Before loading to production, keep one record per order (the first time that order appears) and flag each surviving record.

For every kept order, return all original fields plus two flags:
- `is_valid_email`: 1 when the email looks valid (contains `@`) and 0 otherwise.
- `is_clean`: 1 only when the record passes every quality check at once (valid email, a positive quantity, a present price, and a date that is not in the future) and 0 if any check fails.

**Schema columns:** `raw_orders.order_id`, `raw_orders.customer_email`, `raw_orders.product_name`, `raw_orders.quantity`, `raw_orders.unit_price`, `raw_orders.order_date`, `raw_orders.status`

**Output columns:** `order_id`, `customer_email`, `is_valid_email`, `product_name`, `quantity`, `unit_price`, `order_date`, `status`, `is_clean`

Sort by the first column in ascending order.

### Examples

#### Example 1
**Input**

**raw_orders**

| order_id | customer_email                                    | product_name   | quantity | unit_price | order_date | status    |
| -------: | ------------------------------------------------- | -------------- | -------: | ---------: | ---------- | --------- |
|      101 | [alice@example.com](mailto:alice@example.com)     | Laptop         |        1 |     999.99 | 2024-01-15 | shipped   |
|      102 | [bob@example.com](mailto:bob@example.com)         | Mouse          |        2 |      25.50 | 2024-02-10 | shipped   |
|      101 | [alice@example.com](mailto:alice@example.com)     | Laptop Bag     |        1 |      49.99 | 2024-01-16 | pending   |
|      103 | invalid-email                                     | Monitor        |        1 |     299.99 | 2024-03-05 | shipped   |
|      104 | NULL                                              | Keyboard       |        1 |      79.99 | 2024-03-12 | pending   |
|      105 | [charlie@example.com](mailto:charlie@example.com) | Desk           |       -2 |     199.99 | 2024-04-01 | pending   |
|      106 | [david@example.com](mailto:david@example.com)     | Chair          |        0 |      89.99 | 2024-04-15 | shipped   |
|      107 | [eve@example.com](mailto:eve@example.com)         | Bookshelf      |        2 |       NULL | 2024-05-20 | shipped   |
|      108 | frankexample.com                                  | Lamp           |        1 |      45.00 | 2024-06-01 | delivered |
|      109 | NULL                                              | Table          |        1 |     150.00 | 2024-06-15 | shipped   |
|      110 | [grace@example.com](mailto:grace@example.com)     | Sofa           |        1 |     799.99 | 2025-01-10 | pending   |
|      102 | [bob@example.com](mailto:bob@example.com)         | Wireless Mouse |        1 |      35.00 | 2024-02-11 | delivered |
|      111 | [henry@example.com](mailto:henry@example.com)     | Headphones     |        3 |      59.99 | 2024-07-04 | shipped   |
|      112 | ian@                                              | Webcam         |        1 |      75.00 | 2024-08-18 | shipped   |
|      113 | [jane@example.com](mailto:jane@example.com)       | Tablet         |        1 |     499.99 | 2024-12-31 | delivered |

---

**Expected Output**

| order_id | customer_email                                    | is_valid_email | product_name | quantity | unit_price | order_date | status    | is_clean |
| -------: | ------------------------------------------------- | :------------: | ------------ | -------: | ---------: | ---------- | --------- | :------: |
|      101 | [alice@example.com](mailto:alice@example.com)     |        1       | Laptop       |        1 |     999.99 | 2024-01-15 | shipped   |     1    |
|      102 | [bob@example.com](mailto:bob@example.com)         |        1       | Mouse        |        2 |      25.50 | 2024-02-10 | shipped   |     1    |
|      103 | invalid-email                                     |        0       | Monitor      |        1 |     299.99 | 2024-03-05 | shipped   |     0    |
|      104 | NULL                                              |        0       | Keyboard     |        1 |      79.99 | 2024-03-12 | pending   |     0    |
|      105 | [charlie@example.com](mailto:charlie@example.com) |        1       | Desk         |       -2 |     199.99 | 2024-04-01 | pending   |     0    |
|      106 | [david@example.com](mailto:david@example.com)     |        1       | Chair        |        0 |      89.99 | 2024-04-15 | shipped   |     0    |
|      107 | [eve@example.com](mailto:eve@example.com)         |        1       | Bookshelf    |        2 |       NULL | 2024-05-20 | shipped   |     0    |
|      108 | frankexample.com                                  |        0       | Lamp         |        1 |      45.00 | 2024-06-01 | delivered |     0    |
|      109 | NULL                                              |        0       | Table        |        1 |     150.00 | 2024-06-15 | shipped   |     0    |
|      110 | [grace@example.com](mailto:grace@example.com)     |        1       | Sofa         |        1 |     799.99 | 2025-01-10 | pending   |     0    |
|      111 | [henry@example.com](mailto:henry@example.com)     |        1       | Headphones   |        3 |      59.99 | 2024-07-04 | shipped   |     1    |
|      112 | ian@                                              |        1       | Webcam       |        1 |      75.00 | 2024-08-18 | shipped   |     1    |
|      113 | [jane@example.com](mailto:jane@example.com)       |        1       | Tablet       |        1 |     499.99 | 2024-12-31 | delivered |     1    |


**Explanation:** Order 1 appears twice, so only its first record (Laptop) is kept. Order 6 has a future date (2025-03-01), so `is_clean` is 0.

### Constraints

- An email is valid when it contains an `@` character.
- Quantity must be greater than 0; price must not be missing; the date must be on or before 2024-12-31.
- When the same `order_id` appears multiple times, keep only the first occurrence.
- `is_clean` is 1 only when every check passes.
- Return results matching the expected output schema and order.

In [0]:
raw_orders_data = [(101, "alice@example.com", "Laptop", 1, 999.99, "2024-01-15", "shipped"),(102, "bob@example.com", "Mouse", 2, 25.50, "2024-02-10", "shipped"),(101, "alice@example.com", "Laptop Bag", 1, 49.99, "2024-01-16", "pending"),(103, "invalid-email", "Monitor", 1, 299.99, "2024-03-05", "shipped"),(104, None, "Keyboard", 1, 79.99, "2024-03-12", "pending"),(105, "charlie@example.com", "Desk", -2, 199.99, "2024-04-01", "pending"),(106, "david@example.com", "Chair", 0, 89.99, "2024-04-15", "shipped"),(107, "eve@example.com", "Bookshelf", 2, None, "2024-05-20", "shipped"),(108, "frankexample.com", "Lamp", 1, 45.00, "2024-06-01", "delivered"),(109, None, "Table", 1, 150.00, "2024-06-15", "shipped"),(110, "grace@example.com", "Sofa", 1, 799.99, "2025-01-10", "pending"),(102, "bob@example.com", "Wireless Mouse", 1, 35.00, "2024-02-11", "delivered"),(111, "henry@example.com", "Headphones", 3, 59.99, "2024-07-04", "shipped"),(112, "ian@", "Webcam", 1, 75.00, "2024-08-18", "shipped"),(113, "jane@example.com", "Tablet", 1, 499.99, "2024-12-31", "delivered")]

raw_orders_df = spark.createDataFrame(raw_orders_data,["order_id","customer_email","product_name","quantity","unit_price","order_date","status"])

raw_orders_df = raw_orders_df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

window_spec = Window.partitionBy("order_id").orderBy(col("order_date").desc())

orders_df = raw_orders_df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")

is_clean_condition = (
    (col("is_valid_email") == 1) 
    & (col("quantity") > 0) 
    & (col("order_date").isNotNull())
    & (col("order_date") <= current_date())
    & (col("unit_price").isNotNull())
)


# select: order_id, customer_email, is_valid_email, product_name, quantity, unit_price, order_date, status, is_clean
orders_df = (
orders_df
    .withColumn("is_valid_email", 
        when(col("customer_email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), 1)
        .otherwise(0)
    )    
    .withColumn("is_clean",  when(is_clean_condition, 1).otherwise(0))
    .select("order_id","customer_email", "is_valid_email", "product_name", "quantity", "unit_price", "order_date", "status", "is_clean")
    .orderBy(1)
)


display(orders_df)




## Que3: Removing Duplicates from DataFrame

**Difficulty:** Easy

### Problem

A customer table stores one row every time a customer's profile is updated, so a single customer can appear many times with different signup dates. Produce a deduplicated table that keeps exactly one row per customer: the row carrying that customer's latest `signup_date`. Return `customer_id`, `name`, `email`, and that latest `signup_date`, ordered by `customer_id`.

**Schema columns:** `customer_records.record_id`, `customer_records.customer_id`, `customer_records.name`, `customer_records.email`, `customer_records.signup_date`

**Output columns:** `customer_id`, `name`, `email`, `signup_date`

Order the result by `customer_id`.

### Examples

#### Example 1

**Input:**

**customer_records:**

| record_id | customer_id | name | email | signup_date |
|-----------|------------:|------|-------|-------------|
| R001 | C001 | John Smith | [email protected] | 2024-01-01 |
| R002 | C001 | John Smith | [email protected] | 2024-01-15 |
| R003 | C002 | Jane Doe | [email protected] | 2024-01-25 |
| R011 | C002 | Jane Doe | [email protected] | 2024-01-02 |
| R004 | C003 | Bob Wilson | [email protected] | 2024-01-03 |
| R005 | C003 | Bob Wilson | [email protected] | 2024-01-20 |

**Output:**

| customer_id | name | email | signup_date |
|------------:|------|-------|-------------|
| C001 | John Smith | [email protected] | 2024-01-15 |
| C002 | Jane Doe | [email protected] | 2024-01-25 |
| C003 | Bob Wilson | [email protected] | 2024-01-20 |

**Explanation:** Customer C001 has two rows dated 2024-01-01 and 2024-01-15; the later date 2024-01-15 is kept. C002's rows are 2024-01-25 and 2024-01-02, so 2024-01-25 survives even though it appears first in the input.

### Constraints

- Return exactly one row per `customer_id`.
- Keep the row whose `signup_date` is the most recent for that customer.
- Output only `customer_id`, `name`, `email`, `signup_date` (the `record_id` is not returned).
- Order the result by `customer_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
customer_records_data = [("R001","C001","John Smith","[email protected]","2024-01-01"),("R002","C001","John Smith","[email protected]","2024-01-15"),("R003","C002","Jane Doe","[email protected]","2024-01-25"),("R011","C002","Jane Doe","[email protected]","2024-01-02"),("R004","C003","Bob Wilson","[email protected]","2024-01-03"),("R005","C003","Bob Wilson","[email protected]","2024-01-20")]
customer_records_df = spark.createDataFrame(customer_records_data, ["record_id","customer_id","name","email","signup_date"])

window_spec = Window.partitionBy("customer_id").orderBy(col("signup_date").desc())

output_df = (
customer_records_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
    .select("customer_id","name","email", "signup_date")
    .orderBy("customer_id")
)


display(customer_records_df)
display(output_df)

## Que4: Partition Pruning Optimization Demo

**Difficulty:** Easy

### Problem

The analytics team wants a Q1 2024 summary from the events table. Keep only events whose `event_date` falls on or after `2024-01-01` and strictly before `2024-04-01` (a range covering January through March 2024). For each month-and-category combination in that window, report one row: `partition_month` as the string `'YYYY-MM'`, `category`, `event_count` as the number of events, and `total_value` as the sum of value. Sort by `partition_month` ascending, then `category` ascending.

**Schema columns:** `events.event_id`, `events.event_date`, `events.category`, `events.value`

**Output columns:** `partition_month`, `category`, `event_count`, `total_value`

### Examples

#### Example 1

**Input:**

**events:**

| event_id | event_date | category | value |
|---------:|------------|----------|------:|
| 1 | 2024-01-01 | login | 1 |
| 2 | 2024-01-01 | purchase | 50 |
| 5 | 2024-01-03 | purchase | 30 |
| 6 | 2024-02-01 | login | 1 |
| 8 | 2024-02-02 | click | 1 |
| 14 | 2024-03-03 | purchase | 100 |
| 17 | 2024-04-01 | purchase | 45 |

**Output:**

| partition_month | category | event_count | total_value |
|----------------|----------|------------:|------------:|
| 2024-01 | login | 1 | 1 |
| 2024-01 | purchase | 2 | 80 |
| 2024-02 | click | 1 | 1 |
| 2024-02 | login | 1 | 1 |
| 2024-03 | purchase | 1 | 100 |

**Explanation:** January 2024 has two purchase events (values 50 and 30), so the row (2024-01, purchase) shows `event_count` 2 and `total_value` 80. The purchase on 2024-04-01 is on the range's upper bound and is excluded.

### Constraints

- Keep only `event_date >= 2024-01-01` and `event_date < 2024-04-01` (the upper bound is exclusive).
- `partition_month` must be the string `'YYYY-MM'`.
- One output row per (`partition_month`, `category`) pair present in the filtered data.
- Sort by `partition_month` ascending, then `category` ascending.
- Return results matching the expected output schema and order.

In [0]:
events_data = [(1,"2024-01-01","login",1),(2,"2024-01-01","purchase",50),(5,"2024-01-03","purchase",30),(6,"2024-02-01","login",1),(8,"2024-02-02","click",1),(14,"2024-03-03","purchase",100),(17,"2024-04-01","purchase",45)]
events_df = spark.createDataFrame(events_data, ["event_id","event_date","category","value"])

events_df = (
events_df
    .withColumn("event_date", to_date(col("event_date"), "yyyy-MM-dd"))
    .filter(col("event_date").between("2024-01-01", "2024-03-30") )
    .withColumn("partition_month", date_format(col("event_date"), "yyyy-MM"))
)

display(events_df)

grouped_df = (
events_df.groupBy("partition_month", "category").agg(
    count("event_id").alias("event_count"),
    sum("value").alias("total_value")
)
.orderBy("partition_month", "category")
)

display(grouped_df)



## Que5: Data Validation and Cleaning with Pandas

**Difficulty:** Medium

### Problem

A signup pipeline has ingested raw customer records where some emails are malformed, some ages fall outside a plausible human range, and some salary figures are missing. Without dropping any record, produce a validated and cleaned view.

For each record return its `id`, `name`, and `email`; a flag `is_valid_email` that is 1 when the email contains an `@` character and 0 otherwise; the `signup_date` and `age` exactly as stored; a flag `is_valid_age` that is 1 when age is present and falls between 0 and 120 inclusive and 0 otherwise; and `salary_cleaned`, which keeps the original salary when it is present and otherwise substitutes the median salary across all records that have one.

**Schema columns:** `raw_data.id`, `raw_data.name`, `raw_data.email`, `raw_data.phone`, `raw_data.signup_date`, `raw_data.age`, `raw_data.salary`

**Output columns:** `id`, `name`, `email`, `is_valid_email`, `signup_date`, `age`, `is_valid_age`, `salary_cleaned`

Order the result by `id`.

### Examples

#### Example 1

**Input:**

**raw_data:**

| id | name | email | phone | signup_date | age | salary |
|---:|------|-------|-------|-------------|----:|-------:|
| 1 | John Doe | [email protected] | 555-0001 | 2024-01-01 | 25 | 50000 |
| 2 | Jane Smith | jane.example.com | 555-0002 | 2024-01-02 | 30 | 60000 |
| 3 | Bob Wilson | [email protected] | 555-0003 | 2025-03-15 | -5 | 55000 |
| 4 | Alice Brown | [email protected] | 555-0004 | 2024-01-04 | 150 | 65000 |
| 5 | Charlie White | charlie@ | 555-0005 | 2024-01-05 | 28 | NULL |
| 9 | Grace Lee | [email protected] | 555-0009 | 2024-01-09 | NULL | 50000 |

**Output:**

| id | name | email | is_valid_email | signup_date | age | is_valid_age | salary_cleaned |
|---:|------|-------|:--------------:|-------------|----:|:------------:|---------------:|
| 1 | John Doe | [email protected] | 1 | 2024-01-01 | 25 | 1 | 50000 |
| 2 | Jane Smith | jane.example.com | 0 | 2024-01-02 | 30 | 1 | 60000 |
| 3 | Bob Wilson | [email protected] | 1 | 2025-03-15 | -5 | 0 | 55000 |
| 4 | Alice Brown | [email protected] | 1 | 2024-01-04 | 150 | 0 | 65000 |
| 5 | Charlie White | charlie@ | 1 | 2024-01-05 | 28 | 1 | 55000 |
| 9 | Grace Lee | [email protected] | 1 | 2024-01-09 | NULL | 0 | 50000 |

**Explanation:** Charlie White (id 5) has a missing salary, so `salary_cleaned` becomes the median of the five present salaries (50000, 50000, 55000, 60000, 65000), which is 55000. Grace Lee (id 9) has no age, so `is_valid_age` is 0.

### Constraints

- Valid email: contains an `@` character.
- Valid age: present and between 0 and 120 inclusive.
- Missing salary: replace with the median of all non-missing salaries.
- `signup_date` and `age` are returned unchanged, including future dates and out-of-range ages.
- Return results matching the expected output schema and order.

In [0]:
raw_data_data = [(1,"John Doe","[email protected]","555-0001","2024-01-01",25,50000),(2,"Jane Smith","jane.example.com","555-0002","2024-01-02",30,60000),(3,"Bob Wilson","[email protected]","555-0003","2025-03-15",-5,55000),(4,"Alice Brown","[email protected]","555-0004","2024-01-04",150,65000),(5,"Charlie White","charlie@","555-0005","2024-01-05",28,None),(9,"Grace Lee","[email protected]","555-0009","2024-01-09",None,50000)]
raw_data_df = spark.createDataFrame(raw_data_data, ["id","name","email","phone","signup_date","age","salary"])

display(raw_data_df)

## Que6: Merging DataFrames with Indicator for Join Type Tracking

**Difficulty:** Medium

### Problem

An HR system stores employees in one table and departments in another, linked by `dept_id`. Some employees belong to a department that no longer exists, and some departments currently have no employees. Combine the two tables so that every employee and every department appears at least once, keeping the department details wherever they match.

For each combined record, report `emp_id`, `emp_name`, `dept_name`, `location`, and a `merge_status` column:
- `both`: the employee's department exists in the departments table.
- `left_only`: the employee exists but their department is not found.
- `right_only`: the department exists but has no employee.

Return all records ordered by `emp_id`.

**Schema columns:** `departments.dept_id`, `departments.dept_name`, `departments.location`, `employees.emp_id`, `employees.emp_name`, `employees.dept_id`

**Output columns:** `emp_id`, `emp_name`, `dept_name`, `location`, `merge_status`

Order the result by `emp_id`.

### Examples

#### Example 1

**Input:**

**departments:**

| dept_id | dept_name | location |
|---------|-----------|----------|
| D001 | Engineering | New York |
| D002 | Sales | Los Angeles |
| D003 | HR | Chicago |
| D005 | Finance | Boston |

**employees:**

| emp_id | emp_name | dept_id |
|--------|----------|--------|
| E001 | Alice Smith | D001 |
| E002 | Bob Johnson | D001 |
| E003 | Carol White | D002 |
| E004 | David Brown | D003 |
| E005 | Eve Wilson | D002 |
| E006 | Frank Miller | D004 |

**Output:**

| emp_id | emp_name | dept_name | location | merge_status |
|--------|----------|-----------|----------|-------------|
| E001 | Alice Smith | Engineering | New York | both |
| E002 | Bob Johnson | Engineering | New York | both |
| E003 | Carol White | Sales | Los Angeles | both |
| E004 | David Brown | HR | Chicago | both |
| E005 | Eve Wilson | Sales | Los Angeles | both |
| E006 | Frank Miller | NULL | NULL | left_only |
| NULL | NULL | Finance | Boston | right_only |

**Explanation:** Frank Miller (E006) sits in department D004, which has no matching row in the departments table, so his `dept_name` and `location` are empty and his `merge_status` is `left_only`. Department D005 (Finance, Boston) has no employee assigned, so it appears with empty `emp_id`/`emp_name` and a `merge_status` of `right_only`.

### Constraints

- Every employee and every department must appear at least once.
- Records present on both sides are marked `both`; an employee with no matching department is `left_only`; a department with no employee is `right_only`.
- Missing values from the unmatched side are left empty.
- Order by `emp_id`, with empty `emp_id` values last.
- Return results matching the expected output schema and order.

In [0]:
departments_data = [("D001","Engineering","New York"),("D002","Sales","Los Angeles"),("D003","HR","Chicago"),("D005","Finance","Boston")]
departments_df = spark.createDataFrame(departments_data, ["dept_id","dept_name","location"])

employees_data = [("E001","Alice Smith","D001"),("E002","Bob Johnson","D001"),("E003","Carol White","D002"),("E004","David Brown","D003"),("E005","Eve Wilson","D002"),("E006","Frank Miller","D004")]
employees_df = spark.createDataFrame(employees_data, ["emp_id","emp_name","dept_id"])

display(departments_df)
display(employees_df)

joined_df = (
employees_df.alias("e").join(departments_df.alias("d"), on=col("e.dept_id") == col("d.dept_id"), how="full")
)

output_df = (
joined_df
    .withColumn("merge_status", 
        when((col("e.dept_id").isNotNull()) & (col("d.dept_id").isNotNull()), "both").
        when(col("e.dept_id").isNotNull(), "left_side").
        when(col("d.dept_id").isNotNull(), "right_side")   
    )
    .orderBy(col("emp_id").asc_nulls_last())
)

display(output_df)


## Que7: Window Function on Partitioned Parquet Data

**Difficulty:** Medium

### Problem

A fintech dashboard needs each user's cumulative spend after every transaction. For each input row, return the transaction details and `running_total`, the sum of that user's amounts through the current transaction. Process transactions by date, breaking same-day ties by `transaction_id`, and order the final rows by user, date, and transaction ID.

**Schema columns:** `transactions.transaction_id`, `transactions.user_id`, `transactions.transaction_date`, `transactions.amount`

**Output columns:** `transaction_id`, `user_id`, `transaction_date`, `amount`, `running_total`

### Examples

#### Example 1

**Input:**

**transactions:**

| transaction_id | user_id | transaction_date | amount |
|---------------:|--------:|------------------|-------:|
| 2 | 1 | 2024-01-01 | 50 |
| 1 | 1 | 2024-01-01 | 100 |
| 3 | 1 | 2024-01-02 | 25 |
| 4 | 2 | 2024-01-01 | 80 |
| 5 | 2 | 2024-01-03 | 20 |

**Output:**

| transaction_id | user_id | transaction_date | amount | running_total |
|---------------:|--------:|------------------|-------:|--------------:|
| 1 | 1 | 2024-01-01 | 100 | 100 |
| 2 | 1 | 2024-01-01 | 50 | 150 |
| 3 | 1 | 2024-01-02 | 25 | 175 |
| 4 | 2 | 2024-01-01 | 80 | 80 |
| 5 | 2 | 2024-01-03 | 20 | 100 |

**Explanation:** User 1's same-day transactions are processed by ID, so transaction 1 starts at 100 and transaction 2 adds 50 for a `running_total` of 150; transaction 3 then raises it to 175.

### Constraints

- Use every supplied input row when calculating the result.
- Preserve the calculation, filtering, tie handling, and ordering described in the problem.
- Return results matching the expected output schema and order.

In [0]:
transactions_data = [(2,1,"2024-01-01",50),(1,1,"2024-01-01",100),(3,1,"2024-01-02",25),(4,2,"2024-01-01",80),(5,2,"2024-01-03",20)]
transactions_df = spark.createDataFrame(transactions_data, ["transaction_id","user_id","transaction_date","amount"])

display(transactions_df)

window_spec = Window.partitionBy("user_id").orderBy("transaction_date", "transaction_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

output_df = (
transactions_df
    .withColumn("running_total", sum("amount").over(window_spec))
)

display(output_df)


## Que8: ETL Pipeline Design for Stripe Data

**Difficulty:** Medium

### Problem

A finance team needs a clean extract of successful Stripe payments. Keep one row per payment ID, discard rows with missing amounts, convert cents to currency units, and derive the transaction date after interpreting the stored offset timestamp in UTC.

**Schema columns:** `stripe_payments.payment_id`, `stripe_payments.amount_cents`, `stripe_payments.currency`, `stripe_payments.status`, `stripe_payments.created_at`, `stripe_payments.merchant_id`, `stripe_payments.metadata_json`

**Output columns:** `payment_id`, `amount`, `currency`, `status`, `transaction_date`, `merchant_id`

- Include only `status = 'succeeded'` rows with non-null `amount_cents`, and return one row per distinct payment record.
- `amount` is cents divided by 100 and rounded to 2 decimal places.
- Interpret the offset in `created_at`, convert the timestamp to UTC, and then extract `transaction_date`.
- Order by `payment_id` ascending.

### Examples

#### Example 1

**Input:**

**stripe_payments:**

| payment_id | amount_cents | currency | status | created_at | merchant_id | metadata_json |
|-----------|-------------:|----------|--------|------------|------------|---------------|
| PAY001 | 5000 | USD | succeeded | 2024-01-15T13:30:00.000+05:30 | M001 | meta_001 |
| PAY003 | 7500 | USD | failed | 2024-01-15T16:00:00.000+05:30 | M001 | meta_003 |
| PAY001 | 5000 | USD | succeeded | 2024-01-15T13:30:00.000+05:30 | M001 | meta_001 |
| PAY011 | 3500 | USD | succeeded | 2024-01-16T02:00:00.000+05:30 | M002 | meta_011 |
| PAY015 | NULL | USD | succeeded | 2024-01-16T17:30:00.000+05:30 | M003 | meta_015 |

**Output:**

| payment_id | amount | currency | status | transaction_date | merchant_id |
|-----------|-------:|----------|--------|-----------------|------------|
| PAY001 | 50.0 | USD | succeeded | 2024-01-15 | M001 |
| PAY011 | 35.0 | USD | succeeded | 2024-01-15 | M002 |

**Explanation:** PAY011's local timestamp falls on January 16, but its UTC time is 2024-01-15 20:30, so its `transaction_date` is 2024-01-15.

### Constraints

- Include only `status = 'succeeded'` rows with non-null `amount_cents`, and return one row per distinct payment record.
- `amount` is cents divided by 100 and rounded to 2 decimal places.
- Interpret the offset in `created_at`, convert the timestamp to UTC, and then extract `transaction_date`.
- Order by `payment_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
stripe_payments_data = [("PAY001",5000,"USD","succeeded","2024-01-15T13:30:00.000+05:30","M001","meta_001"),("PAY003",7500,"USD","failed","2024-01-15T16:00:00.000+05:30","M001","meta_003"),("PAY001",5000,"USD","succeeded","2024-01-15T13:30:00.000+05:30","M001","meta_001"),("PAY011",3500,"USD","succeeded","2024-01-16T02:00:00.000+05:30","M002","meta_011"),("PAY015",None,"USD","succeeded","2024-01-16T17:30:00.000+05:30","M003","meta_015")]
stripe_payments_df = spark.createDataFrame(stripe_payments_data, ["payment_id","amount_cents","currency","status","created_at","merchant_id","metadata_json"])

display(stripe_payments_df)

filtered_df = (
stripe_payments_df
    .filter((col("status") == "succeeded") & (col("amount_cents").isNotNull()))
    .dropDuplicates()
)

output_df = (
filtered_df
    .withColumn("amount", round(col("amount_cents") / 100, 2))
    .orderBy("payment_id")
)

display(output_df)


## Que9: String Parsing and JSON-like Extraction

**Difficulty:** Medium

### Problem

A marketplace stores listing metadata as pipe-separated `key=value` pairs. Return each listing with `color` and `size` extracted as text, using an empty string when either key is absent, and `weight_value` extracted as the numeric portion of `weight`, using NULL when no numeric weight is present. Preserve `listing_id`, `product_name`, and `list_date`, and order by `listing_id` ascending.

**Schema columns:** `product_listings.listing_id`, `product_listings.product_name`, `product_listings.metadata`, `product_listings.list_date`

**Output columns:** `listing_id`, `product_name`, `color`, `size`, `weight_value`, `list_date`

### Examples

#### Example 1

**Input:**

**product_listings:**

| listing_id | product_name | metadata | list_date |
|-----------:|-------------|----------|-----------|
| 1 | Blue T-Shirt | color=blue\|size=medium\|weight=0.3kg | 2024-01-10 |
| 2 | Red Dress | color=red\|size=large\|weight=0.5kg | 2024-01-12 |
| 3 | Jeans | color=blue\|weight=0.6kg | 2024-01-15 |
| 4 | Winter Coat | color=black\|size=xl\|weight=2.5kg | 2024-01-18 |
| 5 | Sneakers | color=white\|size=10\|weight=0.4kg | 2024-01-20 |
| 6 | Shorts | color=green\|size=small | 2024-01-22 |
| 7 | Sweater | color=gray\|size=medium\|weight=0.7kg | 2024-01-25 |
| 8 | Socks | color=black\|size=one size\|weight=0.1kg | 2024-01-28 |

**Output:**

| listing_id | product_name | color | size | weight_value | list_date |
|-----------:|-------------|-------|------|-------------:|-----------|
| 1 | Blue T-Shirt | blue | medium | 0.3 | 2024-01-10 |
| 2 | Red Dress | red | large | 0.5 | 2024-01-12 |
| 3 | Jeans | blue | | 0.6 | 2024-01-15 |
| 4 | Winter Coat | black | xl | 2.5 | 2024-01-18 |
| 5 | Sneakers | white | 10 | 0.4 | 2024-01-20 |
| 6 | Shorts | green | small | NULL | 2024-01-22 |
| 7 | Sweater | gray | medium | 0.7 | 2024-01-25 |
| 8 | Socks | black | one size | 0.1 | 2024-01-28 |

**Explanation:** Listing 1's metadata yields color blue, size medium, and numeric weight 0.3; listing 3 demonstrates a missing size as an empty cell, and listing 6 demonstrates a missing weight as NULL.

### Constraints

- Use every supplied input row when calculating the result.
- Preserve the calculation, filtering, tie handling, and ordering described in the problem.
- Return results matching the expected output schema and order.

In [0]:
product_listings_data = [(1,"Blue T-Shirt","color=blue|size=medium|weight=0.3kg","2024-01-10"),(2,"Red Dress","color=red|size=large|weight=0.5kg","2024-01-12"),(3,"Jeans","color=blue|weight=0.6kg","2024-01-15"),(4,"Winter Coat","color=black|size=xl|weight=2.5kg","2024-01-18"),(5,"Sneakers","color=white|size=10|weight=0.4kg","2024-01-20"),(6,"Shorts","color=green|size=small","2024-01-22"),(7,"Sweater","color=gray|size=medium|weight=0.7kg","2024-01-25"),(8,"Socks","color=black|size=one size|weight=0.1kg","2024-01-28")]
product_listings_df = spark.createDataFrame(product_listings_data, ["listing_id","product_name","metadata","list_date"])

display(product_listings_df)


result = (
    product_listings_df
    .withColumn("metadata_parts", split(col("metadata"), r"\|"))
    .withColumn(
        "color",
        coalesce(
            regexp_extract(
                expr("get(filter(metadata_parts, x -> x rlike '^color='), 0)"),
                r"^color=(.*)$",
                1
            ),
            lit("")
        )
    )
    .withColumn(
        "size",
        coalesce(
            regexp_extract(
                expr("get(filter(metadata_parts, x -> x rlike '^size='), 0)"),
                r"^size=(.*)$",
                1
            ),
            lit("")
        )
    )
    .withColumn(
        "weight_value",
        regexp_extract(
            expr("get(filter(metadata_parts, x -> x rlike '^weight='), 0)"),
            r"^weight=([0-9]+(?:\.[0-9]+)?)",
            1
        )
    )
    .withColumn(
        "weight_value",
        when(col("weight_value") == "", lit(None))
         .otherwise(col("weight_value").cast("double"))
    )
    .select(
        "listing_id",
        "product_name",
        "color",
        "size",
        "weight_value",
        "list_date"
    )
    .orderBy("listing_id")
)

display(result)

## Que10: Approximate String Matching

**Difficulty:** Hard

### Problem

A data-quality team must link company records whose names become equal after a small, defined normalization. Compare names without regard to letter case, replace periods with spaces, trim surrounding spaces, and repeatedly remove supported corporate suffixes from the end; label every returned pair `normalized`.

The removable trailing suffixes are `Inc`, `Corporation`, `Corp`, `LLC`, `Ltd`, and `Com`, matched without regard to case; remove adjacent supported suffixes until none remains at the end.

**Schema columns:** `companies_a.company_id`, `companies_a.company_name`, `companies_b.company_id`, `companies_b.company_name`

**Output columns:** `company_a_id`, `company_a_name`, `company_b_id`, `company_b_name`, `match_type`

Order by `company_a_id` ascending.

### Examples

#### Example 1

**Input:**

**companies_a:**

| company_id | company_name |
|-----------:|-------------|
| 1 | Apple Inc |
| 2 | Microsoft Corporation |
| 4 | Amazon Inc |

**companies_b:**

| company_id | company_name |
|-----------:|-------------|
| 101 | Apple Inc |
| 102 | Microsoft Corp |
| 104 | Amazon Incorporated |
| 111 | IBM Corporation |

**Output:**

| company_a_id | company_a_name | company_b_id | company_b_name | match_type |
|-------------:|---------------|-------------:|---------------|------------|
| 1 | Apple Inc | 101 | Apple Inc | normalized |
| 2 | Microsoft Corporation | 102 | Microsoft Corp | normalized |

**Explanation:** Removing the supported suffix from `Microsoft Corporation` and `Microsoft Corp` leaves the same normalized name, while `Incorporated` is not in the removable suffix list.

### Constraints

- Compare names case-insensitively after replacing periods with spaces, trimming surrounding spaces, and repeatedly removing trailing supported suffixes: `Inc`, `Corporation`, `Corp`, `LLC`, `Ltd`, or `Com`.
- Every returned `match_type` is `normalized`.
- Order by `company_a_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
companies_a_data = [(1,"Apple Inc"),(2,"Microsoft Corporation"),(4,"Amazon Inc")]
companies_a_df = spark.createDataFrame(companies_a_data, ["company_id","company_name"])

companies_b_data = [(101,"Apple Inc"),(102,"Microsoft Corp"),(104,"Amazon Incorporated"),(111,"IBM Corporation")]
companies_b_df = spark.createDataFrame(companies_b_data, ["company_id","company_name"])

suffix_pattern = r"(?i)(?:\s+(?:inc|corporation|corp|llc|ltd|com))+\s*$"

def normalize_name(col):
    return trim(
        regexp_replace(

            regexp_replace(lower(col), r"\.", " "),
            
            suffix_pattern,
            ""
        )
    )

a = (
    companies_a_df
    .withColumn("normalized_name", normalize_name(col("company_name")))
)

b = (
    companies_b_df
    .withColumn("normalized_name", normalize_name(col("company_name")))
)

display(a)
display(b)

result = (
    a.alias("a")
    .join(
        b.alias("b"),
        col("a.normalized_name") == col("b.normalized_name"),
        "inner"
    )
    .select(
        col("a.company_id").alias("company_a_id"),
        col("a.company_name").alias("company_a_name"),
        col("b.company_id").alias("company_b_id"),
        col("b.company_name").alias("company_b_name"),
        lit("normalized").alias("match_type")
    )
    .orderBy("company_a_id")
)
display(result)